I reproduce here the model and results of resnet20 architecture on CIFAR-10 dataset, taken from [Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385)

In [1]:
from fastai.vision.all import *
import torch

In [2]:
path_cifar=untar_data(URLs.CIFAR)

<div><progress max="168168549" value="168173568"></progress> 100.00% [168173568/168168549 00:09&lt;00:00]</div>

I apply the same data augmentation: 4 pixels padding per side(final size=40), random cropping of size=32, horizontal flip and noramlization of mean and standard deviation. I also include the transformations that are present by default within the function aug_transforms

In [3]:
dls_cifar=ImageDataLoaders.from_folder(path_cifar,img_cls=PILImage,train='train',valid='test',item_tfms = 
    [Resize(40, method=ResizeMethod.Pad, pad_mode='zeros'),RandomCrop(32)],batch_tfms=[*aug_transforms(),Normalize.from_stats(mean=[0.4925, 0.4758, 0.4386],std=[0.2264, 0.2263, 0.2455])],bs=128,valid_pct=0.1)

from x.mean and x.std i take the mean and standard deviation and include them in Normalize.from_stats
the statistics i obtained are the same ones you see in the cell above for mean and std, while after re-running this cell  i obtain mean around zero, and std around 1, which is the goal of the normalization

In [4]:

x,y=first(dls_cifar.train)
x.mean((0,2,3)),x.std((0,2,3))

(TensorImage([-0.0373, -0.0182, -0.0288], device='cuda:0'),
 TensorImage([1.0024, 0.9697, 0.9427], device='cuda:0'))

In [5]:
def _conv_block(ni,nf,stride):
    return nn.Sequential(
        ConvLayer(ni, nf, stride=stride),
        ConvLayer(nf, nf, act_cls=None, norm_type=NormType.BatchZero))

In [6]:
class ResBlock(Module):
    def __init__(self, ni, nf, stride=1):
        self.convs = _conv_block(ni,nf,stride)
        self.idconv = noop if ni==nf else ConvLayer(ni, nf, 1, act_cls=None)
        self.pool = noop if stride==1 else nn.AvgPool2d(2, ceil_mode=True)

    def forward(self, x):
        return F.relu(self.convs(x) + self.idconv(self.pool(x)))

resnet20 is available built-in from pytorch, but here i will build the architecture from scratch as an exercise

In [7]:
resnet20_=nn.Sequential(
    nn.Conv2d(3,16,kernel_size=3),
    ResBlock(16,16),
    ResBlock(16,16),
    ResBlock(16,32,stride=2),
    ResBlock(32,32),
    ResBlock(32,32),
    ResBlock(32,64,stride=2),
    ResBlock(64,64),
    ResBlock(64,64),
    ResBlock(64,64),
    nn.AdaptiveAvgPool2d(1),
    Flatten(),
    nn.Linear(64,10),
)

In [8]:
resnet20_=Learner(dls_cifar,opt_func=SGD,model=resnet20_,loss_func=nn.CrossEntropyLoss(),metrics=accuracy,lr=0.01,wd=0.0001)

In [9]:
#the final number of epochs has to be 182
lr=0.01
resnet20_.fit_one_cycle(60,0.1,moms=(0.9,0.9,0.9))

epoch,train_loss,valid_loss,accuracy,time
0,2.170049,2.152808,0.223667,00:26
1,2.037451,2.012995,0.260333,00:26
2,1.909837,1.876644,0.310167,00:26
3,1.787879,1.736828,0.349167,00:26
4,1.674439,1.638610,0.385000,00:26
5,1.588538,1.554609,0.420000,00:26
6,1.477810,1.490971,0.453667,00:26
7,1.402494,1.491354,0.465000,00:26
8,1.275638,1.611793,0.466167,00:27
9,1.203486,1.159828,0.575500,00:27


the original paper changed learning rate manually, while i used the automated version fit one cycle for convenience. the fact that accuracy goes down for some epochs is expected behaviour from fit_one_cycle, because the initially high learning rate can cause escape from a local minimum